# NB-12: WanVideoPipeline -- Full System Integration

## Learning Objectives
1. **SYS-01: Pipeline Composition** -- Understand how WanModel (DiT), WanVideoVAE, WanTextEncoder (T5/UMT5-XXL), and WanImageEncoder (CLIP ViT-H/14) compose into the full WanVideoPipeline that generates motion-controlled videos
2. **SYS-02: Tensor Data Flow** -- Trace any tensor from raw inputs (text prompt, reference image, control video) through T5/CLIP encoding, VAE latent encoding, 48-channel concatenation, DiT denoising, and VAE decoding to the output video `[B, 3, F, H, W]`
3. **SYS-03: Parameter Count Breakdown** -- Compute and interpret the parameter count across all four model components (DiT, VAE, T5, CLIP) and understand why the full pipeline requires ~8B parameters

## Prerequisites
- **NB-01**: RMSNorm, sinusoidal embeddings, and the `modulate` function -- these are the building blocks of timestep conditioning
- **NB-02**: QKV projections and multi-head attention layout -- the attention mechanism used in both T5 and CLIP
- **NB-03**: 3D Rotary Position Embeddings (RoPE) -- the spatial position encoding used in DiT
- **NB-04**: Self-attention and cross-attention in DiT blocks -- how text context is injected into the DiT
- **NB-05**: adaLN-Zero modulation -- how timestep conditioning gates each DiT block
- **NB-06**: DiT block -- the full attention + FFN block with adaLN gating
- **NB-07**: Patchify/unpatchify and Head -- how video frames become token sequences and back
- **NB-08**: WanModel end-to-end forward pass -- the full 48-channel DiT including gradient checkpointing
- **NB-09**: CausalConv3d, ResidualBlock, AttentionBlock -- the VAE building blocks
- **NB-10**: VAE Encoder3d -- downsampling video to latents
- **NB-11**: VAE Decoder3d and patchify disambiguation -- upsampling latents to video

## Concept Map
Each prior notebook covers one part of the pipeline:

| Notebook | Pipeline Stage |
|----------|---------------|
| NB-01 | RMSNorm + sinusoidal timestep embedding -- used by T5 (RMSNorm) and DiT (sinusoidal) |
| NB-02 | Attention primitives -- used by T5 text encoder and CLIP image encoder |
| NB-03 | 3D RoPE -- spatial position encoding inside DiT blocks |
| NB-04 | Cross-attention -- how T5 `context` and CLIP `clip_feature` condition the DiT |
| NB-05 | adaLN-Zero -- how FlowMatchScheduler timesteps gate each DiT block |
| NB-06 | DiT block -- the core unit repeated 30 times inside WanModel |
| NB-07 | Patchify/Head/unpatchify -- WanModel's input and output projections |
| NB-08 | WanModel forward pass -- the full DiT end-to-end |
| NB-09 | CausalConv3d + ResBlock + AttentionBlock -- VAE building blocks |
| NB-10 | VAE Encoder3d -- raw video `[B,3,F,H,W]` -> latents `[B,16,T_lat,H/8,W/8]` |
| NB-11 | VAE Decoder3d -- latents `[B,16,T_lat,H/8,W/8]` -> video `[B,3,F,H,W]` |
| **NB-12** | **Full pipeline** -- all components composed into WanVideoPipeline |

In [ ]:
# Source: established in NB-01/NB-08/NB-11, extended for all five pipeline modules
# Loads: wan_video_dit, wan_video_vae, wan_video_text_encoder, wan_video_image_encoder, flow_match
import sys
import types
import importlib.util
import pathlib

# ── Step 1: Find project root ─────────────────────────────────────────────────
# Walk up from notebook location (Course/) to find the directory containing diffsynth/
_here = pathlib.Path().resolve()
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):  # search up to 10 levels
    if (_candidate / 'diffsynth' / 'models' / 'wan_video_dit.py').exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:  # reached filesystem root
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find diffsynth/models/wan_video_dit.py. '
        'Run this notebook from Course/ inside the project checkout.'
    )
print(f'Project root: {PROJECT_ROOT}')

# ── Step 2: Stub modules that cause import errors ─────────────────────────────
# Stub wan_video_camera_controller (not needed for pipeline demos)
# Source: same pattern as NB-08 setup cell
_cam_stub = types.ModuleType('diffsynth.models.wan_video_camera_controller')
_cam_stub.SimpleAdapter = type('SimpleAdapter', (), {'__init__': lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType('diffsynth')
_models_stub = types.ModuleType('diffsynth.models')
_diffusion_stub = types.ModuleType('diffsynth.diffusion')
sys.modules['diffsynth'] = _diffsynth_stub
sys.modules['diffsynth.models'] = _models_stub
sys.modules['diffsynth.models.wan_video_camera_controller'] = _cam_stub
sys.modules['diffsynth.diffusion'] = _diffusion_stub

# Note: tqdm is available in this environment -- do not stub it.
# The transformers library imports tqdm.auto which requires the real tqdm package.

# ── Step 3: Load modules in dependency order (wan_video_dit FIRST) ────────────
# wan_video_image_encoder.py imports flash_attention from wan_video_dit,
# so wan_video_dit must be registered in sys.modules before image_encoder loads.

# 3a. Load wan_video_dit.py
_dit_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_dit.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_dit', _dit_path)
_dit_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_dit'] = _dit_mod
_spec.loader.exec_module(_dit_mod)

# 3b. Load wan_video_vae.py
_vae_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_vae.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_vae', _vae_path)
_vae_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_vae'] = _vae_mod
_spec.loader.exec_module(_vae_mod)

# 3c. Load wan_video_text_encoder.py
_te_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_text_encoder.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_text_encoder', _te_path)
_te_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_text_encoder'] = _te_mod
_spec.loader.exec_module(_te_mod)

# 3d. Load wan_video_image_encoder.py
_ie_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_image_encoder.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_image_encoder', _ie_path)
_ie_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_image_encoder'] = _ie_mod
_spec.loader.exec_module(_ie_mod)

# 3e. Load flow_match.py
_fm_path = PROJECT_ROOT / 'diffsynth' / 'diffusion' / 'flow_match.py'
_spec = importlib.util.spec_from_file_location('diffsynth.diffusion.flow_match', _fm_path)
_fm_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.diffusion.flow_match'] = _fm_mod
_spec.loader.exec_module(_fm_mod)

# ── Step 4: Import symbols ─────────────────────────────────────────────────────
from diffsynth.models.wan_video_dit import WanModel, sinusoidal_embedding_1d
from diffsynth.models.wan_video_vae import WanVideoVAE
from diffsynth.models.wan_video_text_encoder import WanTextEncoder
from diffsynth.models.wan_video_image_encoder import WanImageEncoder
from diffsynth.diffusion.flow_match import FlowMatchScheduler
import torch

print('Setup complete. Five modules loaded:')
print('  WanModel              <- diffsynth/models/wan_video_dit.py')
print('  WanVideoVAE           <- diffsynth/models/wan_video_vae.py')
print('  WanTextEncoder        <- diffsynth/models/wan_video_text_encoder.py')
print('  WanImageEncoder       <- diffsynth/models/wan_video_image_encoder.py')
print('  FlowMatchScheduler    <- diffsynth/diffusion/flow_match.py')

## Section 1: Pipeline Overview

The `WanVideoPipeline` is the orchestrator that composes four model components into a complete video generation system:

| Component | Class | Role |
|-----------|-------|------|
| DiT | `WanModel` (30 blocks) | Denoising backbone -- predicts velocity to remove noise |
| VAE | `WanVideoVAE` | Encodes/decodes video between pixel and latent space |
| T5 | `WanTextEncoder` (UMT5-XXL) | Encodes text prompts to semantic context vectors |
| CLIP | `WanImageEncoder` (ViT-H/14) | Encodes reference images to appearance feature tokens |

This is the capstone notebook that converges the **DiT track** (NB-01 to NB-08) and the **VAE track** (NB-09 to NB-11). Every concept from prior notebooks comes together here: sinusoidal timestep embeddings (NB-01), 3D RoPE (NB-03), cross-attention for text conditioning (NB-04), adaLN-Zero (NB-05), DiT blocks (NB-06), patchify/unpatchify (NB-07), the full DiT forward pass (NB-08), and the VAE encode/decode cycle (NB-10, NB-11).

**WanVideoPipeline unit architecture:** The pipeline orchestrates via five processing units (defined in `wan_video.py` lines 55-82):
- `PromptEmbedder` -- runs T5 and CLIP encoders to produce `context` and `clip_feature`
- `FunControl` -- VAE-encodes the control video into 16-channel latents
- `FunReference` -- VAE-encodes the reference image into 16-channel latents, then zero-pads to match temporal dimension
- `ImageEmbedderCLIP` -- combines control and reference latents as the `y` tensor (`[B, 32, T, H, W]`)
- `NoiseInitializer` -- generates the initial random noise latents `[B, 16, T, H, W]`

See `diffsynth/pipelines/wan_video.py` lines 55-82 for the full unit class definitions. Below we focus on the data flow and component-by-component behavior.

## Full Pipeline ASCII Diagram

The diagram below shows the complete data flow from raw inputs through all five pipeline stages to the output video. Every component box includes notebook back-references showing where each piece was taught.

```
Raw Inputs
    |
    +-- Text Prompt (str)
    |       |
    |       v
    |   WanTextEncoder (T5/UMT5-XXL, 24 layers)         [<- NB-01 RMSNorm, NB-02 Attention]
    |   vocab: 256,384  ->  dim: 4,096
    |   output: context [B, L<=512, 4096]
    |
    +-- Reference Image (PIL.Image)
    |       |
    |       +-->  WanImageEncoder (CLIP ViT-H/14, 32 layers)  [<- NB-02, NB-04]
    |       |    224x224  ->  257 tokens x 1,280 dim
    |       |    output: clip_feature [B, 257, 1280]
    |       |
    |       +-->  WanVideoVAE.encode (for reference latent)   [<- NB-10]
    |            [B, 3, 1, H, W]  ->  [B, 16, 1, H/8, W/8]
    |            output: ref_latents [B, 16, 1, H/8, W/8]
    |
    +-- Control Video (list of frames)
    |       |
    |       +-->  WanVideoVAE.encode                          [<- NB-10]
    |            [B, 3, F, H, W]  ->  [B, 16, T_lat, H/8, W/8]
    |            output: control_latents [B, 16, T_lat, H/8, W/8]
    |
    +-- Noise (Random Normal)
            [B, 16, T_lat, H/8, W/8]
            output: noise_latents [B, 16, T_lat, H/8, W/8]

                    +--------------------------------------------+
                    |   48-Channel Concatenation                 |  [<- NB-08]
                    |   x = noise_latents    [B, 16, T, H/8, W/8]|
                    |   y = cat(ctrl, ref)   [B, 32, T, H/8, W/8]|
                    |   -> [B, 48, T, H/8, W/8] inside WanModel  |
                    +--------------------+-----------------------+
                                         |
                    +--------------------v-----------------------+
                    |   WanModel (DiT, 30 blocks)                |  [<- NB-06 blocks,
                    |   patch_embedding: Conv3d(48,1536,k=1,2,2) |      NB-07 patchify,
                    |   Patchify -> [B, T*H/2*W/2, 1536] tokens  |      NB-08 forward]
                    |                                            |
                    |   Inputs also injected:                    |
                    |   +-- context [B, L, 4096] -> [B, L, 1536] |  [<- NB-04 cross-attn]
                    |   +-- clip_feature -> [B, 257, 1536]        |  [<- NB-02]
                    |   |   prepended to context sequence         |
                    |   +-- t_mod [B, 6, 1536]   (adaLN-Zero)    |  [<- NB-05]
                    |   +-- freqs  (3D RoPE)                      |  [<- NB-03]
                    |                                            |
                    |   x 50 timesteps via FlowMatchScheduler    |
                    |   (velocity prediction + scheduler.step)   |
                    |   FlowMatchScheduler                        |  [<- new in NB-12]
                    +--------------------+-----------------------+
                                         |
                              denoised latents
                              [B, 16, T_lat, H/8, W/8]
                                         |
                    +--------------------v-----------------------+
                    |   WanVideoVAE.decode                        |  [<- NB-11]
                    |   Decoder3d upsamples x8 spatial, x4 temp   |
                    |   T_lat = (F-1)//4 + 1 (e.g. 21 for 81fr)  |
                    +--------------------+-----------------------+
                                         |
                               Output Video
                               [B, 3, F, H, W]
```

**Temporal compression note:** The VAE compresses `F` frames to `T_lat = (F-1)//4 + 1` latent frames.
Examples: 5 frames -> 2 latent frames, 17 frames -> 5, 81 frames -> 21.
Use `(F-1)//4 + 1`, NOT `F//4` -- the latter gives wrong counts (see NB-11 disambiguation).

## Section 2: Component Demos

The next four cells each instantiate one pipeline component with a **reduced configuration** for fast CPU execution (per STD-03: each cell must run in under 5 seconds). The demo configs use tiny vocabularies, fewer layers, or small input tensors while keeping production architecture dimensions.

Each demo:
1. Instantiates the component with reduced config (or full config if it's already fast)
2. Runs a forward pass with dummy inputs
3. Asserts the expected output shape
4. Prints a parameter count comparison (demo vs production)

The four demo cells are self-contained -- you can run them independently after the setup cell.

In [ ]:
# Source: diffsynth/models/wan_video_text_encoder.py, class WanTextEncoder line 212
# DEMO CONFIG: vocab=1000, num_layers=2 (production: vocab=256,384, num_layers=24)
# CRITICAL: Full WanTextEncoder() takes ~35s to instantiate on CPU due to
#   Embedding(256384, 4096) = 1.05B parameters initialized at construction time.

text_encoder = WanTextEncoder(
    vocab=1000,      # reduced from 256,384 for speed
    dim=4096,        # production architecture dimension
    dim_attn=4096,
    dim_ffn=10240,
    num_heads=64,
    num_layers=2,    # reduced from 24 for speed
    num_buckets=32,
)

dummy_ids  = torch.zeros(1, 10, dtype=torch.long)   # [B, L]
dummy_mask = torch.ones(1, 10)                       # [B, L]
with torch.no_grad():
    context = text_encoder(dummy_ids, dummy_mask)    # -> [B, L, 4096]

assert context.shape == (1, 10, 4096), f'Expected (1, 10, 4096), got {context.shape}'
print(f'T5 output shape: {context.shape}')
print(f'  Production: vocab=256,384, num_layers=24 -> ~5.68B params')
print(f'  Demo:       vocab=1,000,  num_layers=2  -> {sum(p.numel() for p in text_encoder.parameters()):,} params')
print()
print('Shape: [B=1, L=10, dim=4096] -- each of 10 tokens becomes a 4096-dim context vector')
print('DiT injects context via cross-attention (NB-04) after projecting 4096->1536')

In [ ]:
# Source: diffsynth/models/wan_video_image_encoder.py, class WanImageEncoder line ~852
# Full CLIP ViT-H/14 -- no reduced config needed (2.4s init + ~0.5s forward < 5s)
# WanImageEncoder wraps clip_xlm_roberta_vit_h_14 with pretrained=False for fast init

image_encoder = WanImageEncoder()

dummy_img = torch.randn(1, 3, 224, 224)                           # [B, 3, 224, 224]
with torch.no_grad():
    clip_feature = image_encoder.encode_image([dummy_img])         # list input -> [B, 257, 1280]

assert clip_feature.shape == (1, 257, 1280), f'Expected (1, 257, 1280), got {clip_feature.shape}'
print(f'CLIP output shape: {clip_feature.shape}')
print(f'  257 tokens = 1 [CLS] token + 256 patch tokens  (224/14 = 16, 16*16 = 256)')
print(f'  Parameters: {sum(p.numel() for p in image_encoder.parameters()):,}')
print()
print('Shape: [B=1, 257 tokens, 1280-dim] -- appearance features from the reference image')
print('DiT prepends clip_feature tokens to the text context sequence (after dim projection 1280->1536)')

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, class WanVideoVAE line 1058
# Full VAE with tiny spatial input -- fast on CPU
# WanVideoVAE wraps VideoVAE_ (1.3B param VAE) with learned mean/std normalization

vae = WanVideoVAE()

dummy_video = torch.randn(1, 3, 5, 32, 32)   # [B, C=3, F=5, H=32, W=32] -- tiny input
with torch.no_grad():
    latents = vae.encode(dummy_video, device='cpu')   # [B, 16, T_lat, H/8, W/8]
    decoded = vae.decode(latents, device='cpu')        # [B, 3, F, H, W]

# Temporal compression: T_lat = (F-1)//4 + 1 = (5-1)//4 + 1 = 2
# Spatial compression: H/8 = 32/8 = 4,  W/8 = 32/8 = 4
assert latents.shape == (1, 16, 2, 4, 4), f'Expected (1, 16, 2, 4, 4), got {latents.shape}'
assert decoded.shape == dummy_video.shape, f'Expected {dummy_video.shape}, got {decoded.shape}'

print(f'VAE encode: {dummy_video.shape} -> {latents.shape}')
print(f'VAE decode: {latents.shape} -> {decoded.shape}')
print(f'  Spatial:  /8  ({dummy_video.shape[3]} -> {dummy_video.shape[3]//8})')
print(f'  Temporal: (F-1)//4+1 ({dummy_video.shape[2]} -> {latents.shape[2]})')
print(f'  Latent channels: 3 -> 16')
print(f'  Parameters: {sum(p.numel() for p in vae.parameters()):,}')

### DiT Forward Pass Demo

The DiT (Diffusion Transformer) is the heart of the denoising process. Readers who completed NB-08 have already seen the full 48-channel `WanModel` forward pass. Here we use a base T2V (text-to-video) configuration with `in_dim=16` and `num_layers=3` (vs production 30), matching the same reduced-config approach.

**Key difference from NB-08:** In the full Fun-Control pipeline the model uses `in_dim=48` (48 = 16 noise + 16 ctrl + 16 ref). In this demo we use `in_dim=16` with `has_image_input=False` to demonstrate the base DiT forward pass cleanly. This is the same configuration used in base Wan2.1-T2V.

**Source:** `diffsynth/models/wan_video_dit.py`, `WanModel.__init__` line 274

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, WanModel.__init__ line 274
# DEMO CONFIG: num_layers=3 (production: 30), in_dim=16 (base T2V config)
# Verified: 0.042s on CPU for this config (STD-03 compliant)

dit = WanModel(
    dim=1536,            # production architecture dimension
    in_dim=16,           # base T2V config (production Fun-Control uses in_dim=48)
    ffn_dim=8960,        # production FFN dimension
    out_dim=16,          # output: 16 latent channels
    text_dim=4096,       # T5 text encoder output dim
    freq_dim=256,        # sinusoidal timestep embedding dim
    eps=1e-6,
    patch_size=(1, 2, 2),
    num_heads=12,
    num_layers=3,        # REDUCED from 30 for CPU demo
    has_image_input=False,  # base T2V config -- no CLIP image path
)

# Dummy inputs matching DiT contract
B, F, H, W = 1, 4, 8, 8               # small spatial dims for CPU demo
x = torch.randn(B, 16, F, H, W)        # noise latents [B, in_dim, F, H, W]
timestep = torch.tensor([500.0])        # single timestep value [B]
ctx = torch.randn(B, 10, 4096)          # text encoder output [B, L, 4096]

with torch.no_grad():
    dit.eval()
    out = dit(x, timestep, ctx)          # -> [B, out_dim, F, H, W]

assert out.shape == torch.Size([B, 16, F, H, W]), f'Expected {(B, 16, F, H, W)}, got {out.shape}'
print(f'DiT input:  {x.shape}     (noise latents)')
print(f'DiT output: {out.shape}   (velocity prediction -- direction to clean image)')
print(f'  Demo (3 layers, in_dim=16):   {sum(p.numel() for p in dit.parameters()):,} params')
print(f'  Production (30 layers, in_dim=16): ~1.42B params (base T2V)')
print(f'  Production (30 layers, in_dim=48): ~1.56B params (Fun-Control)')

## How Component Outputs Connect

Each component's output feeds directly into the next stage. Referring back to the pipeline diagram in Cell 3:

| Producer | Output | Consumer | How it enters |
|----------|--------|----------|---------------|
| `WanTextEncoder` | `context [B, L, 4096]` | `WanModel` | `text_embedding` MLP projects 4096 -> 1536; consumed by cross-attention in each DiT block (NB-04) |
| `WanImageEncoder` | `clip_feature [B, 257, 1280]` | `WanModel` | `img_emb` MLP projects 1280 -> 1536; **prepended** to context sequence before DiT attention |
| `WanVideoVAE.encode` | `control_latents [B, 16, T, H/8, W/8]` | `WanModel.forward` | Combined with ref_latents as `y=[B,32,T,H/8,W/8]`; WanModel concatenates `x` and `y` internally to form the 48-channel input |
| `WanVideoVAE.encode` | `ref_latents [B, 16, 1, H/8, W/8]` | `WanModel.forward` | Same as above |
| `WanModel` | `velocity [B, 16, T, H/8, W/8]` | `FlowMatchScheduler.step` | `prev = latents + velocity * (sigma_next - sigma_current)` -- since sigma decreases each step, delta is negative, moving latents toward clean image |
| `FlowMatchScheduler` | updated `latents [B, 16, T, H/8, W/8]` | (repeated 50 times) | After 50 steps, latents are denoised |
| `WanVideoVAE.decode` | `output_video [B, 3, F, H, W]` | (final output) | Decoder3d upsamples x8 spatial, x4 temporal |

**Anti-pattern reminder:** Do NOT pass a pre-concatenated 48-channel tensor to `WanModel.forward`. The method signature is:
```python
def forward(self, x, timestep, context, clip_feature=None, y=None, ...):
```
where `x=[B,16,T,H,W]` (noise latents) and `y=[B,32,T,H,W]` (control+ref latents). WanModel concatenates them internally when `has_image_input=True`.

Source: `diffsynth/models/wan_video_dit.py`, WanModel.forward line 359

## Section 3: 48-Channel Input Composition

The Fun-Control variant of WanModel (the one in this repository) conditions on **two additional signals** beyond the noise latents:
- A **control video** latent (VAE-encoded from the input control video)
- A **reference image** latent (VAE-encoded single frame, zero-padded to match temporal dimension)

These are combined into a 48-channel input using channel-wise concatenation. You first saw this pattern in **NB-08** where the full WanModel forward pass was explored.

### How the concatenation is assembled

The pipeline (`diffsynth/pipelines/wan_video.py`, `model_fn_wan_video`) assembles the inputs as two tensors:

```
x = noise_latents             [B, 16, T, H/8, W/8]   -- the noisy video latents being denoised
y = cat(control, ref_emb)     [B, 32, T, H/8, W/8]   -- 16ch control + 16ch reference
```

Then inside `WanModel.forward` (`wan_video_dit.py` line 359), when `has_image_input=True`:
```python
x = torch.cat([x, y], dim=1)  # -> [B, 48, T, H/8, W/8]
```
This 48-channel tensor is fed into `patch_embedding`, a `Conv3d(48, 1536, kernel_size=(1,2,2))` that tokenizes the video into a sequence of 1536-dim patch tokens.

**CRITICAL:** `WanModel.forward` takes `x=[B,16,T,H,W]` and `y=[B,32,T,H,W]` **separately**. The model performs the concatenation internally. Do NOT pass a pre-concatenated 48-channel tensor to `forward()`.

```python
# Correct call signature (wan_video_dit.py line 359):
def forward(self, x, timestep, context, clip_feature=None, y=None, ...):
    # When has_image_input=True: x = torch.cat([x, y], dim=1)  <- done inside forward
```

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, model_fn_wan_video
# and wan_video_dit.py line 359 -- WanModel.forward does the cat internally

B, C, T, H, W = 1, 16, 4, 8, 8  # latent dimensions (tiny for demo)

noise_latents   = torch.randn(B, C, T, H, W)   # noisy video latent
control_latents = torch.randn(B, C, T, H, W)   # VAE-encoded control video
ref_embedding   = torch.randn(B, C, T, H, W)   # VAE ref + zero-padded temporal

# model_fn_wan_video assembles y = cat(control, ref_emb):
y = torch.cat([control_latents, ref_embedding], dim=1)   # [B, 32, T, H, W]
# Then WanModel.forward does: x = cat([noise_latents, y], dim=1) -> [B, 48, T, H, W]
# We show the assembly but do NOT pass pre-cat to forward -- the model handles it
assert y.shape == (B, 32, T, H, W), f'Expected (1, 32, 4, 8, 8), got {y.shape}'
print(f'noise x:         {noise_latents.shape}  (16 channels)')
print(f'y (ctrl+ref):    {y.shape}  (32 channels)')
print(f'Combined in-model: [B, {noise_latents.shape[1]+y.shape[1]}, T, H, W]  (48 channels -> patch_embedding)')
print()
print('WanModel.forward signature (wan_video_dit.py line 359):')
print('  forward(self, x, timestep, context, clip_feature=None, y=None, ...)')
print('  x=[B,16,T,H,W]  y=[B,32,T,H,W]  -> cat performed INSIDE forward when has_image_input=True')

## Section 4: The Denoising Loop

### What is flow matching?

You may be familiar with DDPM-style diffusion, where the model learns to predict the **noise** that was added to an image. Flow matching is different: the model predicts the **velocity** -- the direction pointing from noise toward clean data.

Formally, if `x_data` is the clean image and `x_noise` is pure Gaussian noise, the flow matching training target is:
```
velocity = x_data - x_noise
```
At inference, we start from pure noise (`sigma=1.0`) and iteratively move toward the clean image (`sigma=0.0`) by following the velocity field.

### The sigma schedule

`FlowMatchScheduler` creates a sequence of decreasing sigma values. For 50 steps with `shift=5.0` (Wan's default):
- `sigma[0] = 1.0` -- pure noise
- `sigma[49] ≈ 0.093` -- nearly clean
- The `shift` parameter redistributes the time budget: higher shift allocates more steps to high-noise (early) denoising

### The step formula

Each denoising step applies:
```python
# Source: diffsynth/diffusion/flow_match.py, line 158
prev_sample = sample + model_output * (sigma_next - sigma_current)
```
Since `sigma_next < sigma_current`, the delta `(sigma_next - sigma_current)` is **negative**.
The velocity points from noise toward data, so multiplying by a negative delta moves the sample **toward the clean manifold**.

This is the key insight: each step says "follow the velocity a little bit in the direction of clean data."

### Classifier-Free Guidance (CFG)

CFG is a technique to steer generation toward the text prompt. The DiT is run **twice** per denoising step:
1. **Positive pass**: use the real text prompt (`context = positive_context`)
2. **Negative pass**: use an empty/negative prompt (`context = negative_context`)

The two predictions are combined:
```python
# Source: diffsynth/pipelines/wan_video.py, lines 301-309
noise_pred_posi = model_fn(..., context=positive_context, ...)
noise_pred_nega = model_fn(..., context=negative_context, ...)
noise_pred = noise_pred_nega + cfg_scale * (noise_pred_posi - noise_pred_nega)
```

- `cfg_scale=1.0`: collapses to `noise_pred = noise_pred_posi` (no guidance)
- `cfg_scale=7.5`: typical Wan2.1 setting -- amplifies the positive prediction relative to the negative
- Higher `cfg_scale` steers generation more strongly toward the prompt (but too high causes artifacts)

The cells below demonstrate the scheduler setup and a minimal 3-step denoising loop.

In [ ]:
# Source: diffsynth/diffusion/flow_match.py, set_timesteps_wan (lines 30-39)
# Demonstrates the sigma schedule for 50-step Wan denoising

scheduler = FlowMatchScheduler('Wan')
scheduler.set_timesteps(
    num_inference_steps=50,    # production uses 50 steps
    denoising_strength=1.0,    # 1.0 = full denoising from noise
    shift=5.0,                 # Wan default -- redistributes time budget
)
print(f'Number of steps: {len(scheduler.timesteps)}')
print(f'First 5 timesteps: {scheduler.timesteps[:5].tolist()}')
print(f'Last 5 timesteps:  {scheduler.timesteps[-5:].tolist()}')
print(f'First 5 sigmas:    {scheduler.sigmas[:5].tolist()}')
print(f'Last 5 sigmas:     {scheduler.sigmas[-5:].tolist()}')
print()
print('sigma = timestep / 1000')
print('sigma=1.0 -> pure noise, sigma=0.0 -> clean image')
print(f'Range: {scheduler.sigmas[0]:.4f} (noisy) -> {scheduler.sigmas[-1]:.4f} (nearly clean)')

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 290-314
# Demo: 3 steps with base T2V DiT (has_image_input=False), simplified
# Uses the 'dit' instance from Cell 9 (already in memory)

scheduler_demo = FlowMatchScheduler('Wan')
scheduler_demo.set_timesteps(3, denoising_strength=1.0, shift=5.0)

B, F_lat, H_lat, W_lat = 1, 4, 8, 8
latents = torch.randn(B, 16, F_lat, H_lat, W_lat)  # start from pure noise
ctx_demo = torch.randn(B, 10, 4096)  # dummy text context

print('Denoising loop (3 steps, base T2V DiT, no CFG for clarity):')
print(f'{"Step":<6} {"timestep":>10} {"sigma":>8} {"sigma_next":>10} {"latent_norm":>12}')
print('-' * 52)

dit.eval()
with torch.no_grad():
    for i, timestep in enumerate(scheduler_demo.timesteps):
        t = timestep.unsqueeze(0)

        # In production: run DiT twice (positive + negative prompt) for CFG
        # Here we show a single pass for clarity
        noise_pred = dit(latents, t, ctx_demo)  # velocity prediction

        # Update latents using flow matching step
        # Internally: prev_sample = sample + model_output * (sigma_next - sigma)
        latents = scheduler_demo.step(noise_pred, timestep, latents)

        sigma = scheduler_demo.sigmas[i].item()
        sigma_next = scheduler_demo.sigmas[i+1].item() if i+1 < len(scheduler_demo.sigmas) else 0.0
        print(f'{i:<6} {timestep.item():>10.1f} {sigma:>8.4f} {sigma_next:>10.4f} {latents.norm().item():>12.4f}')

print()
print(f'Final latents shape: {latents.shape}')
assert latents.shape == torch.Size([B, 16, F_lat, H_lat, W_lat]), \
    f'Expected [{B}, 16, {F_lat}, {H_lat}, {W_lat}], got {latents.shape}'
print(f'Note: sigma decreases each step (1.0 -> 0.0), delta is negative,')
print(f'      so velocity * negative_delta moves latents toward clean image')

### Classifier-Free Guidance in the Production Denoising Loop

In the demo above we ran a single DiT forward pass per step (no CFG) for simplicity. In production (`wan_video.py` lines 301-309), the loop runs the DiT **twice** per step:

```python
# Source: diffsynth/pipelines/wan_video.py, lines 301-309
noise_pred_posi = model_fn(latents, t, context=positive_context, ...)  # real prompt
noise_pred_nega = model_fn(latents, t, context=negative_context, ...)  # empty/negative prompt

# CFG combination:
cfg_scale = 7.5  # typical Wan2.1 value
noise_pred = noise_pred_nega + cfg_scale * (noise_pred_posi - noise_pred_nega)
```

Understanding this formula:
- When `cfg_scale=1.0`: `noise_pred = noise_pred_nega + 1.0 * (pos - nega) = noise_pred_posi` (only positive)
- When `cfg_scale=7.5`: the difference `(pos - nega)` is amplified 7.5×, strongly steering toward the prompt
- The negative prompt acts as a baseline: "don't go in this direction", amplified away from it

After combining, the velocity `noise_pred` is passed to `scheduler.step()` as before:
```python
latents = scheduler.step(noise_pred, timestep, latents)
```

This is the last piece needed to understand the full denoising loop. Each of the 50 production steps:
1. Runs DiT twice (positive + negative)
2. Combines with CFG
3. Updates latents via the flow matching step formula

After 50 steps, the denoised latents `[B, 16, T_lat, H/8, W/8]` are passed to `WanVideoVAE.decode` to reconstruct the output video.

## Section 5: Parameter Count Breakdown

How large is the full WanVideoPipeline? In NB-06 we analyzed parameter counts at the DiT block level to understand LoRA coverage. Here we extend that approach (same `sum(p.numel() for p in model.parameters())` pattern) to the entire pipeline.

We compute **two tables**:
1. **Live demo counts** -- using the reduced-config instances instantiated earlier in this notebook (fast CPU execution)
2. **Production counts** -- verified via live instantiation of full-scale models on 2026-04-24 (see RESEARCH.md)

**Important correction:** The file `model_architecture.md` in this repository understates the DiT parameter count. The measured value for the Fun-Control variant (in_dim=48, 30 layers) is **1.56B parameters** (1,564,602,176). The production counts below use the verified measured values from RESEARCH.md.

In [ ]:
# Source: NB-06 parameter counting approach, adapted for full pipeline
# Live computation using demo-config models from cells above

demo_components = {
    'DiT (3-block demo, in_dim=16)':  dit,
    'VAE':                             vae,
    'T5 (2-layer demo)':               text_encoder,
    'CLIP (ViT-H/14, full)':           image_encoder,
}

print('Demo parameter counts (reduced configs for CPU execution):')
print('-' * 60)
demo_total = 0
for name, model in demo_components.items():
    n = sum(p.numel() for p in model.parameters())
    demo_total += n
    print(f'  {name:<35}: {n:>15,}')
print(f'  {"-"*35}  {"-"*15}')
print(f'  {"DEMO TOTAL":<35}: {demo_total:>15,}')
print()
print('Note: demo T5 uses vocab=1000, num_layers=2 vs production vocab=256,384, num_layers=24')
print('Note: demo DiT uses num_layers=3 vs production num_layers=30')

In [ ]:
# Production parameter counts -- VERIFIED via live instantiation (see RESEARCH.md)
# NOTE: model_architecture.md understates the DiT param count. Measured values below
# are authoritative (live-computed 2026-04-24). See RESEARCH.md for verification.

production_counts = {
    'DiT (Fun-Control, in_dim=48, 30 layers)': 1_564_602_176,
    'VAE (Encoder3d + Decoder3d)':              126_892_531,
    'T5 Text Encoder (UMT5-XXL, 24 layers)':   5_680_910_336,
    'CLIP Image Encoder (ViT-H/14)':            632_076_801,
}
total = sum(production_counts.values())

print('Production parameter counts (full-size models):')
print('=' * 70)
for name, count in production_counts.items():
    pct = count / total * 100
    billions = count / 1e9
    print(f'  {name:<45}: {count:>15,}  ({pct:4.1f}%)  [{billions:.2f}B]')
print(f'  {"-"*45}  {"-"*15}  {"-"*7}')
print(f'  {"TOTAL":<45}: {total:>15,}          [{total/1e9:.2f}B]')
print()
print('Key observations:')
print(f'  - T5 text encoder dominates at {production_counts["T5 Text Encoder (UMT5-XXL, 24 layers)"]/total*100:.0f}% of total params')
print(f'  - DiT is {production_counts["DiT (Fun-Control, in_dim=48, 30 layers)"]/total*100:.0f}% -- the "1.3B" product name likely refers to DiT+VAE+CLIP without T5')
print(f'  - VAE is compact at {production_counts["VAE (Encoder3d + Decoder3d)"]/total*100:.0f}% despite handling full video reconstruction')

assert total == 8_004_481_844, f'Expected 8,004,481,844, got {total}'

## Summary

### Key Takeaways

- **Pipeline composition**: WanVideoPipeline orchestrates four model components: T5 text encoder (context `[B, L, 4096]`), CLIP image encoder (visual features `[B, 257, 1280]`), VAE (video <-> latent space), and DiT (denoising in latent space `[B, 16, T, H/8, W/8]`)
- **48-channel input**: The Fun-Control DiT variant assembles `y = cat(control[16], ref[16])` into `[B,32,T,H,W]` and calls `WanModel.forward(x=[B,16,...], y=[B,32,...])`. The model concatenates them internally to form the 48-channel input processed by `patch_embedding Conv3d(48->1536)`
- **Flow matching denoising**: Unlike DDPM noise prediction, flow matching predicts velocity (direction from noise to data). Each step applies `latents += velocity * (sigma_next - sigma_current)` where the negative delta moves toward the clean image
- **Classifier-Free Guidance**: Two DiT passes (positive + negative prompt) combined as `nega + scale * (posi - nega)` to steer generation toward the text prompt
- **Scale**: The full pipeline is ~8.0B parameters, dominated by T5 (5.68B, 71%). The DiT at 1.56B is 20% of total.

### Source References

| Symbol | Location |
|--------|----------|
| `WanVideoPipeline` | `diffsynth/pipelines/wan_video.py`, line 32 |
| `WanVideoPipeline.__call__` | `diffsynth/pipelines/wan_video.py`, line 178 |
| `model_fn_wan_video` | `diffsynth/pipelines/wan_video.py`, line 1127 |
| `WanModel` | `diffsynth/models/wan_video_dit.py`, line 273 |
| `WanModel.forward` | `diffsynth/models/wan_video_dit.py`, line 359 |
| `WanVideoVAE` | `diffsynth/models/wan_video_vae.py`, line 1058 |
| `WanTextEncoder` | `diffsynth/models/wan_video_text_encoder.py`, line 212 |
| `WanImageEncoder` | `diffsynth/models/wan_video_image_encoder.py`, line 852 |
| `FlowMatchScheduler` | `diffsynth/diffusion/flow_match.py`, line 5 |
| `FlowMatchScheduler.step` | `diffsynth/diffusion/flow_match.py`, line 149 |

### Course Map

This notebook converges everything from NB-01 through NB-11:

| Notebook | Taught | Used in NB-12 for |
|----------|--------|-------------------|
| **NB-01** | RMSNorm, sinusoidal embeddings, modulate | Timestep conditioning in DiT (sinusoidal); RMSNorm in T5 |
| **NB-02** | QKV projections, multi-head attention | Attention modules in DiT blocks, T5, CLIP |
| **NB-03** | 3D RoPE | Spatial/temporal position encoding inside DiT blocks |
| **NB-04** | Self-attention, cross-attention | How T5 `context` and CLIP `clip_feature` condition the DiT |
| **NB-05** | adaLN-Zero modulation | How FlowMatchScheduler timesteps gate each DiT block |
| **NB-06** | Full DiT block | The core unit repeated 30 times inside WanModel |
| **NB-07** | Patchify, unpatchify, Head | WanModel's video-to-token conversion |
| **NB-08** | WanModel forward pass | The full 30-block DiT end-to-end |
| **NB-09** | CausalConv3d, ResBlock, AttentionBlock | VAE building blocks for temporal-causal 3D convolution |
| **NB-10** | VAE Encoder3d | Video `[B,3,F,H,W]` -> latents `[B,16,T_lat,H/8,W/8]` |
| **NB-11** | VAE Decoder3d | Latents `[B,16,T_lat,H/8,W/8]` -> video `[B,3,F,H,W]` |
| **NB-12** | Full WanVideoPipeline | This notebook -- all components composed |


## Exercises

### Exercise 1 -- Change DiT Block Count

In Cell 9, change `num_layers=3` to `num_layers=5`. Re-run Cell 9 and then Cell 18.

1. How does the demo DiT parameter count change?
2. Use the per-block count from NB-06 (~51.2M params/block) to **predict** the new total before running.
3. What is the difference between 3-block and 5-block DiT parameter counts? Is it exactly 2 × per-block count?

*Hint: Check NB-06 Cell 12 for the per-block breakdown. The extra parameters beyond `30 × per_block` in the production DiT come from head, embedding, and layer-norm modules.*

---

### Exercise 2 -- Explore the Sigma Schedule

In Cell 14, experiment with the scheduler settings:

1. Change `num_inference_steps=50` to `num_inference_steps=10`. How does the sigma spacing change? Are the first/last sigma values the same?
2. Change `shift=5.0` to `shift=1.0`. Compare the first 5 sigmas. What does a lower shift do to the time budget allocation?
3. Change `shift=5.0` to `shift=10.0`. How does the sigma distribution change relative to `shift=5.0`?

*Hint: Higher shift concentrates more denoising steps in the high-sigma (high-noise) regime. The shift parameter in `set_timesteps_wan` applies `sigmas = shift * sigmas / (1 + (shift - 1) * sigmas)` -- try plotting `sigmas` with `import matplotlib.pyplot as plt; plt.plot(scheduler.sigmas)`.*

---

### Exercise 3 -- CFG Scale Analysis

The CFG formula is: `noise_pred = noise_pred_nega + cfg_scale * (noise_pred_posi - noise_pred_nega)`

Without running any code, answer these questions:

1. If `cfg_scale=0.0`, what does `noise_pred` equal? What effect would this have on generation?
2. If `cfg_scale=1.0`, what does `noise_pred` equal? (Simplify the formula.)
3. If `noise_pred_posi == noise_pred_nega` (positive and negative predictions are identical), what does `cfg_scale` do?
4. Why might `cfg_scale=20.0` cause visual artifacts in generated videos?

*Hint: Think about what happens when you amplify a small difference very strongly -- the generation becomes extremely sensitive to any deviation from the negative prompt baseline.*